# House Prices — v5: Metaheuristic hyperparameter optimization comparison

Builds on v4 (GPU-maximized stack: CatBoost + SVR + Kernel Ridge, with statistical significance
testing).

**Changes in this version:**
- **Added six metaheuristic optimization algorithms** — Genetic Algorithm (GA), Particle Swarm
  Optimization (PSO), Ant Colony Optimization (ACO), Artificial Bee Colony (ABC), Whale
  Optimization Algorithm (WOA), and Simulated Annealing (SA) — each implemented from scratch
  to tune XGBoost hyperparameters over the same search space Optuna uses.
- **Fair comparison loop**: every algorithm (including Optuna as baseline) runs under the same
  function-evaluation budget, and we record wall-clock time, best CV RMSE, and convergence
  history for each.
- **Analysis**: convergence plots, a summary table, paired significance tests against the Optuna
  baseline, and a final verdict on whether metaheuristic tuning helps, hurts, or is neutral
  compared to Optuna's TPE sampler on this problem.
- All v4 features (CatBoost GPU, SVR, Kernel Ridge, statistical significance testing, stacking)
  are unchanged.


In [ ]:
# ===== Setup =====
import warnings
warnings.filterwarnings("ignore")

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import skew, ttest_rel, wilcoxon, friedmanchisquare

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold, train_test_split
from sklearn.linear_model import RidgeCV, ElasticNetCV
from sklearn.kernel_ridge import KernelRidge
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error
from sklearn.utils import check_random_state
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import multiprocessing
N_JOBS = multiprocessing.cpu_count()
print(f"Parallelism: {N_JOBS} logical CPUs")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Libraries that usually need an explicit install on a fresh machine.
# Uncomment if needed:
# %pip install -q xgboost catboost optuna mlflow torch

import xgboost as xgb
import catboost as cb
import torch          # only used to detect a CUDA GPU for the xgb/catboost device flags below --
                       # v4 has no NN base learner, so torch.nn is no longer needed.
import optuna

optuna.logging.set_verbosity(optuna.logging.WARNING)

# MLflow is optional: it's only used for experiment tracking, never for the modeling itself. On an
# offline machine (e.g. a Kaggle Notebook with internet access disabled) "pip install mlflow" can't
# reach PyPI, so we fall back to a no-op shim with the same API surface -- every mlflow.* call below
# then does nothing instead of raising, and the rest of the pipeline runs unaffected.
try:
    import mlflow
    MLFLOW_AVAILABLE = True
except ImportError:
    MLFLOW_AVAILABLE = False
    class _NoOpRun:
        def __enter__(self):
            return self
        def __exit__(self, *exc):
            return False
    class _NoOpMlflow:
        def set_tracking_uri(self, *a, **k): pass
        def set_experiment(self, *a, **k): pass
        def start_run(self, *a, **k): return _NoOpRun()
        def log_params(self, *a, **k): pass
        def log_param(self, *a, **k): pass
        def log_metric(self, *a, **k): pass
        def log_artifact(self, *a, **k): pass
    mlflow = _NoOpMlflow()
    print("mlflow unavailable (not installed / no internet) -- tracking disabled, pipeline runs normally")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
XGB_DEVICE = "cuda" if DEVICE.type == "cuda" else "cpu"
CATBOOST_TASK_TYPE = "GPU" if DEVICE.type == "cuda" else "CPU"

print("xgboost:", xgb.__version__, "| catboost:", cb.__version__, "| optuna:", optuna.__version__,
      "| mlflow:", (mlflow.__version__ if MLFLOW_AVAILABLE else "unavailable (shim)"),
      "| device:", DEVICE)


In [ ]:
# ===== Experiment / tuning configuration =====
# Flip this off to skip all Optuna searches and just reuse FALLBACK_PARAMS (fast rerun).
RUN_OPTUNA = True

# v5: also controls whether the metaheuristic comparison section runs.
RUN_METAHEURISTICS = True

# Optuna trial budgets per model. XGBoost and CatBoost are GPU-accelerated so a single fit is
# cheap -- their budgets are raised versus v3 to use the freed-up compute. SVR/Kernel Ridge have
# no GPU-accelerated build here (that would need cuML/RAPIDS -- a heavier dependency we're not
# pulling in for two base models) but are cheap to fit regardless at ~1,460 rows.
N_TRIALS_XGB = 40
N_TRIALS_CATBOOST = 40
N_TRIALS_ID3 = 20
N_TRIALS_SVR = 25
N_TRIALS_KRR = 25

# v5: metaheuristic evaluation budget. Each algorithm gets this many objective function calls
# (i.e. repeated_cv_score evaluations). Set equal to N_TRIALS_XGB for a fair head-to-head
# comparison against Optuna on the XGBoost search space.
N_EVALS_METAHEURISTIC = N_TRIALS_XGB

# Objective scoring: repeated K-fold (multiple seeds averaged) instead of one noisy split.
TUNING_N_SPLITS = 3      # folds per repeat during search (smaller than final for speed)
TUNING_N_REPEATS = 2     # independent shuffles of the fold assignment, averaged
TUNING_SEEDS = [RANDOM_STATE + i * 1000 for i in range(TUNING_N_REPEATS)]

# Final OOF generation (used for stacking + the submission) uses a larger, multi-seed setup.
FINAL_N_SPLITS = 5
FINAL_SEEDS = [RANDOM_STATE, RANDOM_STATE + 1, RANDOM_STATE + 2]

# MLflow: local SQLite-backed tracking, no server needed.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("house_prices_v5")

FALLBACK_PARAMS = {
    "xgb": dict(n_estimators=2000, learning_rate=0.02, max_depth=3, subsample=0.7,
                 colsample_bytree=0.7, reg_alpha=0.05, reg_lambda=0.0, min_child_weight=2),
    "catboost": dict(iterations=1500, learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
                       bagging_temperature=0.5, random_strength=1.0),
    "id3": dict(n_estimators=60, max_depth=10, min_samples_split=10,
                 min_samples_leaf=3, max_features="sqrt"),
    "svr": dict(C=10.0, epsilon=0.01, gamma=1e-3, kernel="rbf"),
    "krr": dict(alpha=1.0, gamma=1e-3, kernel="rbf"),
}


In [ ]:
# ===== Load data =====
# Auto-detects Kaggle's mounted competition data (/kaggle/input/...) so this cell works unmodified
# both as a Kaggle Notebook and locally; falls back to a local "data/" folder otherwise.
_KAGGLE_DIR = "/kaggle/input/house-prices-advanced-regression-techniques"
DATA_DIR = _KAGGLE_DIR if os.path.exists(_KAGGLE_DIR) else "data"
print("DATA_DIR:", DATA_DIR)

train = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

print("train:", train.shape, " test:", test.shape)
train.head()

## Quick EDA

`SalePrice` is right-skewed (a small number of very expensive houses), so we model
`log1p(SalePrice)` instead of the raw price. This matches the competition's own leaderboard
metric (RMSE on log price).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(train["SalePrice"], bins=50, color="#4C72B0")
axes[0].set_title("SalePrice (raw)")
axes[1].hist(np.log1p(train["SalePrice"]), bins=50, color="#55A868")
axes[1].set_title("log1p(SalePrice)")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(train["GrLivArea"], train["SalePrice"], alpha=0.5, color="#4C72B0")
ax.axvline(4000, color="#C44E52", linestyle="--", label="GrLivArea = 4000")
ax.set_xlabel("GrLivArea")
ax.set_ylabel("SalePrice")
ax.set_title("The two classic Ames outliers")
ax.legend()
plt.tight_layout()
plt.show()

missing = train.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0].head(20)
plt.figure(figsize=(9, 4))
missing.plot(kind="bar", color="#C44E52")
plt.title("Top 20 columns by missing values (train)")
plt.ylabel("# missing")
plt.tight_layout()
plt.show()

## Outlier removal

Two houses in the training set have `GrLivArea` > 4000 sqft but a comparatively low `SalePrice`
(partial sales or otherwise unusual). The dataset's original author (Dean De Cock) explicitly
flags these as safe to drop, and every strong public solution for this competition removes them.
This only touches `train` — never drop rows from `test`.

In [ ]:
outlier_mask = (train["GrLivArea"] > 4000) & (train["SalePrice"] < 300000)
print(f"Dropping {int(outlier_mask.sum())} known GrLivArea outliers")
train = train.loc[~outlier_mask].reset_index(drop=True)
print("train after outlier removal:", train.shape)

## NA semantics: "missing" vs "doesn't have this feature"

For columns like `PoolQC`, `Alley`, `FireplaceQu`, `GarageType`, or `BsmtQual`, `NaN` almost always
means the house doesn't have a pool / alley access / fireplace / garage / basement, not that the
value is unknown. Categorical "absence" columns get filled with the literal string `"None"`,
numeric "absence" columns get filled with `0`.

In [ ]:
NA_MEANS_NONE_CATEGORICAL = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType",
]
NA_MEANS_ZERO_NUMERIC = [
    "GarageYrBlt", "GarageArea", "GarageCars",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath", "MasVnrArea",
]

def fix_na_semantics(df):
    df = df.copy()
    for col in NA_MEANS_NONE_CATEGORICAL:
        if col in df.columns:
            df[col] = df[col].fillna("None")
    for col in NA_MEANS_ZERO_NUMERIC:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    return df

## Ordinal-encode the quality columns

In [ ]:
_QUALITY_SCALE = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
_BSMT_EXPOSURE_SCALE = {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4}
_BSMT_FINTYPE_SCALE = {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6}

ORDINAL_QUALITY_MAPS = {
    col: _QUALITY_SCALE
    for col in ["ExterQual", "ExterCond", "BsmtQual", "BsmtCond", "HeatingQC",
                "KitchenQual", "FireplaceQu", "GarageQual", "GarageCond", "PoolQC"]
}
ORDINAL_QUALITY_MAPS["BsmtExposure"] = _BSMT_EXPOSURE_SCALE
ORDINAL_QUALITY_MAPS["BsmtFinType1"] = _BSMT_FINTYPE_SCALE
ORDINAL_QUALITY_MAPS["BsmtFinType2"] = _BSMT_FINTYPE_SCALE

def ordinal_encode_quality(df):
    df = df.copy()
    for col, mapping in ORDINAL_QUALITY_MAPS.items():
        if col in df.columns:
            df[col] = df[col].fillna("None").map(mapping)
            df[col] = df[col].fillna(0)
    return df

ORDINAL_COLS = list(ORDINAL_QUALITY_MAPS.keys())

## Rare-category grouping

In [ ]:
RARE_CATEGORY_THRESHOLD = 0.01

def group_rare_categories(X_train, X_test, exclude_cols, threshold=RARE_CATEGORY_THRESHOLD):
    X_train, X_test = X_train.copy(), X_test.copy()
    cat_cols = [c for c in X_train.select_dtypes(exclude=[np.number]).columns
                if c not in exclude_cols]
    grouped_cols = []
    for col in cat_cols:
        freq = X_train[col].value_counts(normalize=True)
        rare_levels = freq[freq < threshold].index
        if len(rare_levels) == 0:
            continue
        X_train[col] = X_train[col].where(~X_train[col].isin(rare_levels), "RareOther")
        X_test[col] = X_test[col].where(X_test[col].isin(freq.index) & ~X_test[col].isin(rare_levels),
                                         "RareOther")
        grouped_cols.append(col)
    print(f"Grouped rare levels in {len(grouped_cols)} categorical columns: {grouped_cols}")
    return X_train, X_test

## Feature engineering, target transform & out-of-fold Neighborhood encoding

In [ ]:
def engineer_features(df):
    df = df.copy()
    df["TotalSF"] = df["TotalBsmtSF"].fillna(0) + df["1stFlrSF"] + df["2ndFlrSF"]
    df["HouseAge"] = df["YrSold"] - df["YearBuilt"]
    df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]
    df["TotalBath"] = (df["FullBath"].fillna(0) + 0.5 * df["HalfBath"].fillna(0)
                        + df["BsmtFullBath"].fillna(0) + 0.5 * df["BsmtHalfBath"].fillna(0))
    df["HasPool"] = (df["PoolArea"].fillna(0) > 0).astype(int)
    df["Has2ndFloor"] = (df["2ndFlrSF"].fillna(0) > 0).astype(int)
    df["HasGarage"] = (df["GarageArea"].fillna(0) > 0).astype(int)
    df["HasBsmt"] = (df["TotalBsmtSF"].fillna(0) > 0).astype(int)
    df["HasFireplace"] = (df["Fireplaces"].fillna(0) > 0).astype(int)
    df["QualxSF"] = df["OverallQual"] * df["TotalSF"]
    return df

y = np.log1p(train["SalePrice"])
X = train.drop(columns=["SalePrice", "Id"])
X_test = test.drop(columns=["Id"])
test_ids = test["Id"].values

for fn in (fix_na_semantics, ordinal_encode_quality, engineer_features):
    X = fn(X)
    X_test = fn(X_test)

neighborhood_train = X["Neighborhood"].values
neighborhood_test = X_test["Neighborhood"].values

X, X_test = group_rare_categories(X, X_test, exclude_cols=["Neighborhood"])

print(X.shape, X_test.shape)

## Skew correction on the numeric features

In [ ]:
_SKIP_SKEW_FIX = set(ORDINAL_COLS) | {
    "HasPool", "Has2ndFloor", "HasGarage", "HasBsmt", "HasFireplace",
    "MSSubClass", "OverallQual", "OverallCond", "MoSold", "YrSold",
}

def fix_skew(X, X_test, threshold=0.75):
    numeric_cols = [c for c in X.select_dtypes(include=[np.number]).columns
                     if c not in _SKIP_SKEW_FIX]
    skewed = X[numeric_cols].apply(lambda col: skew(col.dropna()))
    skewed_cols = skewed[skewed.abs() > threshold].index.tolist()
    print(f"log1p-transforming {len(skewed_cols)} skewed numeric features")
    X, X_test = X.copy(), X_test.copy()
    for col in skewed_cols:
        X[col] = np.log1p(X[col].clip(lower=0))
        X_test[col] = np.log1p(X_test[col].clip(lower=0))
    return X, X_test, skewed_cols

X, X_test, skewed_cols = fix_skew(X, X_test)

## Preprocessing pipeline

In [ ]:
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"{len(num_cols)} numeric, {len(cat_cols)} categorical columns")

preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

X_proc = preprocessor.fit_transform(X)
X_test_proc = preprocessor.transform(X_test)

X_proc = X_proc.toarray() if hasattr(X_proc, "toarray") else X_proc
X_test_proc = X_test_proc.toarray() if hasattr(X_test_proc, "toarray") else X_test_proc
y_arr = y.values

print("Final feature matrix (pre target-encoding):", X_proc.shape, "| test:", X_test_proc.shape)

## Out-of-fold target encoding helper

In [ ]:
def kfold_target_encode(train_vals, y_train, apply_vals, smoothing=10.0):
    df = pd.DataFrame({"cat": train_vals, "y": y_train})
    global_mean = float(np.mean(y_train))
    stats = df.groupby("cat")["y"].agg(["mean", "count"])
    smoothed_map = (stats["mean"] * stats["count"] + global_mean * smoothing) / (stats["count"] + smoothing)
    encoded = pd.Series(apply_vals).map(smoothed_map).fillna(global_mean).values.astype(np.float64)
    return encoded.reshape(-1, 1)

## Repeated, multi-seed OOF evaluation

In [ ]:
def _kfold_oof_once(model_fn, X, y, X_test, neigh_train, neigh_test, n_splits, seed, name=""):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    test_fold_preds = np.zeros((n_splits, len(X_test)))
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        te_tr = kfold_target_encode(neigh_train[tr_idx], y[tr_idx], neigh_train[tr_idx])
        te_val = kfold_target_encode(neigh_train[tr_idx], y[tr_idx], neigh_train[val_idx])
        te_test = kfold_target_encode(neigh_train[tr_idx], y[tr_idx], neigh_test)

        X_tr = np.hstack([X[tr_idx], te_tr])
        X_val = np.hstack([X[val_idx], te_val])
        X_te = np.hstack([X_test, te_test])

        model = model_fn(random_state=seed + fold)
        model.fit(X_tr, y[tr_idx])
        oof[val_idx] = model.predict(X_val)
        test_fold_preds[fold] = model.predict(X_te)
    return oof, test_fold_preds.mean(axis=0)


def _oof_seed_worker(seed, model_fn, X, y, X_test, neigh_train, neigh_test, n_splits, name):
    return _kfold_oof_once(model_fn, X, y, X_test, neigh_train, neigh_test, n_splits, seed, name)


def get_oof_predictions(model_fn, X, y, X_test, neigh_train, neigh_test,
                         n_splits=FINAL_N_SPLITS, seeds=FINAL_SEEDS, name="", verbose=True):
    results = Parallel(n_jobs=min(len(seeds), N_JOBS), backend="threading")(
        delayed(_oof_seed_worker)(seed, model_fn, X, y, X_test, neigh_train, neigh_test, n_splits, name)
        for seed in tqdm(seeds, desc=f"OOF seeds [{name}]", leave=False)
    )
    oof_runs, test_runs, rmses = [], [], []
    for seed, (oof, test_pred) in zip(seeds, results):
        rmse = np.sqrt(mean_squared_error(y, oof))
        rmses.append(rmse)
        oof_runs.append(oof)
        test_runs.append(test_pred)
        if verbose:
            print(f"  [{name}] seed {seed}: OOF RMSE(log) = {rmse:.4f}")
    oof_avg = np.mean(oof_runs, axis=0)
    test_avg = np.mean(test_runs, axis=0)
    avg_rmse = np.sqrt(mean_squared_error(y, oof_avg))
    if verbose:
        print("  -> [%s] mean per-seed RMSE = %.4f +/- %.4f | RMSE of averaged OOF = %.4f" %
              (name, np.mean(rmses), np.std(rmses), avg_rmse))
    return oof_avg, test_avg, rmses, oof_runs


def _cv_seed_worker(seed, model_fn, X, y, neigh_train, n_splits):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    oof = np.zeros(len(X))
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        te_tr = kfold_target_encode(neigh_train[tr_idx], y[tr_idx], neigh_train[tr_idx])
        te_val = kfold_target_encode(neigh_train[tr_idx], y[tr_idx], neigh_train[val_idx])
        X_tr = np.hstack([X[tr_idx], te_tr])
        X_val = np.hstack([X[val_idx], te_val])
        model = model_fn(random_state=seed + fold)
        model.fit(X_tr, y[tr_idx])
        oof[val_idx] = model.predict(X_val)
    return float(np.sqrt(mean_squared_error(y, oof)))


def repeated_cv_score(model_fn, X, y, neigh_train,
                       n_splits=TUNING_N_SPLITS, seeds=TUNING_SEEDS, inner_n_jobs=None):
    _jobs = inner_n_jobs if inner_n_jobs is not None else min(len(seeds), N_JOBS)
    rmses = Parallel(n_jobs=_jobs, backend="threading")(
        delayed(_cv_seed_worker)(seed, model_fn, X, y, neigh_train, n_splits)
        for seed in tqdm(seeds, desc="CV seeds", leave=False)
    )
    return float(np.mean(rmses)), float(np.std(rmses))


## Model 1 — XGBoost with Lasso (L1) regularization, Optuna-tuned

In [ ]:
def build_xgb(params, random_state=RANDOM_STATE):
    return xgb.XGBRegressor(
        objective="reg:squarederror",
        random_state=random_state,
        n_jobs=-1,
        tree_method="hist",
        device=XGB_DEVICE,
        **params,
    )

def xgb_objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 500, 4000, step=250),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        max_depth=trial.suggest_int("max_depth", 2, 6),
        subsample=trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-4, 1.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-4, 1.0, log=True),
        min_child_weight=trial.suggest_int("min_child_weight", 1, 8),
    )
    mean_rmse, std_rmse = repeated_cv_score(
        lambda random_state: build_xgb(params, random_state), X_proc, y_arr, neighborhood_train)
    with mlflow.start_run(run_name=f"xgb_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("mean_rmse", mean_rmse)
        mlflow.log_metric("std_rmse", std_rmse)
    return mean_rmse

if RUN_OPTUNA:
    with mlflow.start_run(run_name="xgb_tuning"):
        xgb_study = optuna.create_study(direction="minimize",
                                         sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        xgb_study.optimize(xgb_objective, n_trials=N_TRIALS_XGB, show_progress_bar=True)
        best_xgb_params = xgb_study.best_params
        mlflow.log_params({f"best_{k}": v for k, v in best_xgb_params.items()})
        mlflow.log_metric("best_rmse", xgb_study.best_value)
    print("Best XGBoost params:", best_xgb_params)
    print("Best repeated-CV RMSE(log):", xgb_study.best_value)
else:
    best_xgb_params = FALLBACK_PARAMS["xgb"]
    print("RUN_OPTUNA=False, using fallback XGBoost params:", best_xgb_params)


## Model 2 — CatBoost (GPU)

In [ ]:
def build_catboost(params, random_state=RANDOM_STATE):
    return cb.CatBoostRegressor(
        loss_function="RMSE",
        random_seed=random_state,
        task_type=CATBOOST_TASK_TYPE,
        devices="0" if CATBOOST_TASK_TYPE == "GPU" else None,
        verbose=False,
        **params,
    )

def catboost_objective(trial):
    params = dict(
        iterations=trial.suggest_int("iterations", 500, 3000, step=250),
        learning_rate=trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        depth=trial.suggest_int("depth", 4, 10),
        l2_leaf_reg=trial.suggest_float("l2_leaf_reg", 1e-2, 10.0, log=True),
        bagging_temperature=trial.suggest_float("bagging_temperature", 0.0, 1.0),
        random_strength=trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
    )
    mean_rmse, std_rmse = repeated_cv_score(
        lambda random_state: build_catboost(params, random_state), X_proc, y_arr, neighborhood_train)
    with mlflow.start_run(run_name=f"catboost_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("mean_rmse", mean_rmse)
        mlflow.log_metric("std_rmse", std_rmse)
    return mean_rmse

if RUN_OPTUNA:
    with mlflow.start_run(run_name="catboost_tuning"):
        catboost_study = optuna.create_study(direction="minimize",
                                              sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        catboost_study.optimize(catboost_objective, n_trials=N_TRIALS_CATBOOST,
                                 show_progress_bar=True, n_jobs=1)
        best_catboost_params = catboost_study.best_params
        mlflow.log_params({f"best_{k}": v for k, v in best_catboost_params.items()})
        mlflow.log_metric("best_rmse", catboost_study.best_value)
    print("Best CatBoost params:", best_catboost_params)
    print("Best repeated-CV RMSE(log):", catboost_study.best_value)
else:
    best_catboost_params = FALLBACK_PARAMS["catboost"]
    print("RUN_OPTUNA=False, using fallback CatBoost params:", best_catboost_params)


### CatBoost learning curve across boosting iterations

In [ ]:
te_diag = kfold_target_encode(neighborhood_train, y_arr, neighborhood_train)
X_diag = np.hstack([X_proc, te_diag])
X_tr_d, X_val_d, y_tr_d, y_val_d = train_test_split(
    X_diag, y_arr, test_size=0.2, random_state=RANDOM_STATE)

diag_model = build_catboost(best_catboost_params, random_state=RANDOM_STATE)
diag_model.fit(X_tr_d, y_tr_d, eval_set=(X_val_d, y_val_d), use_best_model=False)

val_rmse_curve = diag_model.get_evals_result()["validation"]["RMSE"]
plt.figure(figsize=(6, 4))
plt.plot(val_rmse_curve)
plt.axvline(len(val_rmse_curve) - 1, color="gray", linestyle="--", alpha=0.5)
plt.xlabel("Boosting iteration")
plt.ylabel("Validation RMSE (log)")
plt.title("CatBoost learning curve (single 80/20 split, tuned params)")
plt.tight_layout()
plt.show()


## Model 3 — Random forest of ID3-style decision trees, Optuna-tuned

In [ ]:
class ID3RegressionTree(BaseEstimator, RegressorMixin):
    def __init__(self, max_depth=8, min_samples_split=10, min_samples_leaf=5,
                 n_candidate_features=None, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.n_candidate_features = n_candidate_features
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64)
        self.rng_ = check_random_state(self.random_state)
        self.tree_ = self._grow(X, y, depth=0)
        self._compile_flat()
        return self

    def _compile_flat(self):
        nodes = []
        idx_map = {}
        def _collect(node):
            i = len(nodes)
            idx_map[id(node)] = i
            nodes.append(node)
            if "feat" in node:
                _collect(node["left"])
                _collect(node["right"])
        _collect(self.tree_)
        n = len(nodes)
        self._feat  = np.full(n, -1, dtype=np.int32)
        self._thr   = np.zeros(n, dtype=np.float64)
        self._left  = np.full(n, -1, dtype=np.int32)
        self._right = np.full(n, -1, dtype=np.int32)
        self._val   = np.zeros(n, dtype=np.float64)
        for node in nodes:
            i = idx_map[id(node)]
            self._val[i] = node["value"]
            if "feat" in node:
                self._feat[i]  = node["feat"]
                self._thr[i]   = node["thr"]
                self._left[i]  = idx_map[id(node["left"])]
                self._right[i] = idx_map[id(node["right"])]

    @staticmethod
    def _variance_reduction(y, y_left, y_right):
        if len(y_left) == 0 or len(y_right) == 0:
            return -np.inf
        n = len(y)
        parent_var = np.var(y)
        weighted = (len(y_left) / n) * np.var(y_left) + (len(y_right) / n) * np.var(y_right)
        return parent_var - weighted

    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        feat_idx = np.arange(n_features)
        if self.n_candidate_features:
            k = min(self.n_candidate_features, n_features)
            feat_idx = self.rng_.choice(n_features, size=k, replace=False)
        best_gain, best_feat, best_thr = -np.inf, None, None
        for f in feat_idx:
            col = X[:, f]
            thresholds = np.unique(col)
            if len(thresholds) > 32:
                thresholds = np.unique(np.quantile(col, np.linspace(0.05, 0.95, 32)))
            for thr in thresholds:
                left = col <= thr
                right = ~left
                if left.sum() < self.min_samples_leaf or right.sum() < self.min_samples_leaf:
                    continue
                gain = self._variance_reduction(y, y[left], y[right])
                if gain > best_gain:
                    best_gain, best_feat, best_thr = gain, f, thr
        return best_feat, best_thr, best_gain

    def _grow(self, X, y, depth):
        node = {"value": float(np.mean(y)), "n": len(y)}
        if depth >= self.max_depth or len(y) < self.min_samples_split or np.all(y == y[0]):
            return node
        feat, thr, gain = self._best_split(X, y)
        if feat is None or gain <= 1e-12:
            return node
        left_mask = X[:, feat] <= thr
        node.update({
            "feat": feat, "thr": thr,
            "left": self._grow(X[left_mask], y[left_mask], depth + 1),
            "right": self._grow(X[~left_mask], y[~left_mask], depth + 1),
        })
        return node

    def predict(self, X):
        X = np.asarray(X, dtype=np.float64)
        n = X.shape[0]
        node_idx = np.zeros(n, dtype=np.int32)
        while True:
            feat = self._feat[node_idx]
            is_leaf = feat < 0
            if is_leaf.all():
                break
            go_left = X[np.arange(n), np.where(feat >= 0, feat, 0)] <= self._thr[node_idx]
            next_idx = np.where(go_left, self._left[node_idx], self._right[node_idx])
            node_idx = np.where(is_leaf, node_idx, next_idx)
        return self._val[node_idx]


def _fit_one_tree(X, y, max_depth, min_samples_split, min_samples_leaf, k, seed):
    rng = check_random_state(seed)
    n_samples = X.shape[0]
    idx = rng.randint(0, n_samples, n_samples)
    tree = ID3RegressionTree(
        max_depth=max_depth, min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf, n_candidate_features=k,
        random_state=rng.randint(0, 1_000_000),
    )
    tree.fit(X[idx], y[idx])
    return tree


class ID3RandomForestRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, n_estimators=60, max_depth=10, min_samples_split=10,
                 min_samples_leaf=3, max_features="sqrt", random_state=RANDOM_STATE):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float64)
        y = np.asarray(y, dtype=np.float64)
        rng = check_random_state(self.random_state)
        n_features = X.shape[1]
        if self.max_features == "sqrt":
            k = max(1, int(np.sqrt(n_features)))
        elif self.max_features == "log2":
            k = max(1, int(np.log2(n_features)))
        else:
            k = n_features
        seeds = [rng.randint(0, 1_000_000) for _ in range(self.n_estimators)]
        self.trees_ = Parallel(n_jobs=N_JOBS, backend="loky")(
            delayed(_fit_one_tree)(X, y, self.max_depth, self.min_samples_split,
                                   self.min_samples_leaf, k, s)
            for s in tqdm(seeds, desc="ID3 trees", leave=False)
        )
        return self

    def predict(self, X):
        preds = np.column_stack([t.predict(X) for t in self.trees_])
        return preds.mean(axis=1)


def build_id3_rf(params, random_state=RANDOM_STATE):
    return ID3RandomForestRegressor(random_state=random_state, **params)

def id3_objective(trial):
    params = dict(
        n_estimators=trial.suggest_int("n_estimators", 50, 300, step=25),
        max_depth=trial.suggest_int("max_depth", 4, 16),
        min_samples_split=trial.suggest_int("min_samples_split", 5, 30),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 10),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2"]),
    )
    mean_rmse, std_rmse = repeated_cv_score(
        lambda random_state: build_id3_rf(params, random_state), X_proc, y_arr, neighborhood_train)
    with mlflow.start_run(run_name=f"id3_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("mean_rmse", mean_rmse)
        mlflow.log_metric("std_rmse", std_rmse)
    return mean_rmse

if RUN_OPTUNA:
    with mlflow.start_run(run_name="id3_tuning"):
        id3_study = optuna.create_study(direction="minimize",
                                         sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        id3_study.optimize(id3_objective, n_trials=N_TRIALS_ID3, show_progress_bar=True, n_jobs=1)
        best_id3_params = id3_study.best_params
        mlflow.log_params({f"best_{k}": v for k, v in best_id3_params.items()})
        mlflow.log_metric("best_rmse", id3_study.best_value)
    print("Best ID3 forest params:", best_id3_params)
    print("Best repeated-CV RMSE(log):", id3_study.best_value)
else:
    best_id3_params = FALLBACK_PARAMS["id3"]
    print("RUN_OPTUNA=False, using fallback ID3 forest params:", best_id3_params)


## Model 4 — ElasticNetCV

In [ ]:
def build_elasticnet(random_state=RANDOM_STATE):
    return ElasticNetCV(
        l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 1.0],
        alphas=np.logspace(-4, 1, 50),
        cv=5,
        max_iter=20000,
        random_state=random_state,
    )

## Model 5 — Support Vector Regression (RBF kernel), Optuna-tuned

In [ ]:
def build_svr(params, random_state=RANDOM_STATE):
    return SVR(**params)

def svr_objective(trial):
    params = dict(
        kernel="rbf",
        C=trial.suggest_float("C", 0.1, 100.0, log=True),
        epsilon=trial.suggest_float("epsilon", 1e-3, 1.0, log=True),
        gamma=trial.suggest_float("gamma", 1e-5, 1e-1, log=True),
    )
    mean_rmse, std_rmse = repeated_cv_score(
        lambda random_state: build_svr(params, random_state), X_proc, y_arr, neighborhood_train)
    with mlflow.start_run(run_name=f"svr_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("mean_rmse", mean_rmse)
        mlflow.log_metric("std_rmse", std_rmse)
    return mean_rmse

if RUN_OPTUNA:
    with mlflow.start_run(run_name="svr_tuning"):
        svr_study = optuna.create_study(direction="minimize",
                                         sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        svr_study.optimize(svr_objective, n_trials=N_TRIALS_SVR, show_progress_bar=True)
        best_svr_params = dict(svr_study.best_params)
        best_svr_params["kernel"] = "rbf"
        mlflow.log_params({f"best_{k}": v for k, v in best_svr_params.items()})
        mlflow.log_metric("best_rmse", svr_study.best_value)
    print("Best SVR params:", best_svr_params)
    print("Best repeated-CV RMSE(log):", svr_study.best_value)
else:
    best_svr_params = FALLBACK_PARAMS["svr"]
    print("RUN_OPTUNA=False, using fallback SVR params:", best_svr_params)


## Model 6 — Kernel Ridge Regression (RBF kernel)

In [ ]:
def build_kernelridge(params, random_state=RANDOM_STATE):
    return KernelRidge(**params)

def krr_objective(trial):
    params = dict(
        kernel="rbf",
        alpha=trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        gamma=trial.suggest_float("gamma", 1e-5, 1e-1, log=True),
    )
    mean_rmse, std_rmse = repeated_cv_score(
        lambda random_state: build_kernelridge(params, random_state), X_proc, y_arr, neighborhood_train)
    with mlflow.start_run(run_name=f"krr_trial_{trial.number}", nested=True):
        mlflow.log_params(params)
        mlflow.log_metric("mean_rmse", mean_rmse)
        mlflow.log_metric("std_rmse", std_rmse)
    return mean_rmse

if RUN_OPTUNA:
    with mlflow.start_run(run_name="krr_tuning"):
        krr_study = optuna.create_study(direction="minimize",
                                         sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        krr_study.optimize(krr_objective, n_trials=N_TRIALS_KRR, show_progress_bar=True)
        best_krr_params = dict(krr_study.best_params)
        best_krr_params["kernel"] = "rbf"
        mlflow.log_params({f"best_{k}": v for k, v in best_krr_params.items()})
        mlflow.log_metric("best_rmse", krr_study.best_value)
    print("Best Kernel Ridge params:", best_krr_params)
    print("Best repeated-CV RMSE(log):", krr_study.best_value)
else:
    best_krr_params = FALLBACK_PARAMS["krr"]
    print("RUN_OPTUNA=False, using fallback Kernel Ridge params:", best_krr_params)


---

# Metaheuristic hyperparameter optimization — GA, PSO, ACO, ABC, WOA, SA (v5)

This section implements **six nature-inspired metaheuristic algorithms** from scratch and
benchmarks them against Optuna's TPE sampler on the XGBoost hyperparameter search space.

**Why XGBoost as the benchmark model?** It has the richest search space in this notebook (8
continuous/integer hyperparameters, a mix of linear and log scales), and Optuna's XGBoost result
from above serves as a direct baseline. Running every metaheuristic × every base model would be
prohibitively expensive.

**Fair comparison:** Every algorithm — including Optuna — gets the same evaluation budget
(`N_EVALS_METAHEURISTIC` calls to `repeated_cv_score`). We track wall-clock time, best RMSE
found, and convergence history (best-so-far after each evaluation).

### The six algorithms

| Algorithm | Inspiration | Key mechanism |
|-----------|-------------|---------------|
| **GA** (Genetic Algorithm) | Darwinian evolution | Selection → crossover → mutation over a population of candidate solutions |
| **PSO** (Particle Swarm Optimization) | Bird flocking / fish schooling | Particles update velocity toward personal & global bests |
| **ACO** (Ant Colony Optimization, continuous variant) | Ant pheromone trails | Archive of good solutions sampled with Gaussian kernels |
| **ABC** (Artificial Bee Colony) | Honeybee foraging | Employed → onlooker → scout bee phases |
| **WOA** (Whale Optimization Algorithm) | Humpback whale bubble-net hunting | Encircling prey + spiral update + exploration |
| **SA** (Simulated Annealing) | Metal annealing | Accept worse solutions probabilistically at high temperature, cool down over time |


In [ ]:
# ===== Search-space abstraction =====
# All metaheuristics work in a normalised [0,1]^d hypercube and decode to actual parameter values
# via this helper.  This keeps the algorithms themselves simple and parameter-type-agnostic.

class HyperparameterSpace:
    """Defines a search space as a list of (name, low, high, scale, dtype, step) tuples.
    
    - scale: 'linear' or 'log'  (log for learning_rate, reg_alpha, etc.)
    - dtype: 'float' or 'int'
    - step:  if not None, round integers to nearest multiple of step (e.g. n_estimators step=250)
    """
    def __init__(self, param_defs):
        self.param_defs = param_defs
        self.dim = len(param_defs)
    
    def decode(self, x):
        """Map a vector in [0,1]^d to the actual hyperparameter dict."""
        result = {}
        for i, (name, low, high, scale, dtype, step) in enumerate(self.param_defs):
            v = np.clip(x[i], 0.0, 1.0)
            if scale == "log":
                v = np.exp(np.log(low) + v * (np.log(high) - np.log(low)))
            else:
                v = low + v * (high - low)
            if dtype == "int":
                if step is not None and step > 1:
                    v = int(round(v / step) * step)
                else:
                    v = int(round(v))
                v = max(int(low), min(int(high), v))
            result[name] = v
        return result


# XGBoost search space — identical ranges to the Optuna objective above.
XGB_SPACE = HyperparameterSpace([
    # (name,              low,    high,  scale,    dtype,  step)
    ("n_estimators",      500,    4000,  "linear", "int",  250),
    ("learning_rate",     0.005,  0.1,   "log",    "float", None),
    ("max_depth",         2,      6,     "linear", "int",  1),
    ("subsample",         0.5,    1.0,   "linear", "float", None),
    ("colsample_bytree", 0.5,    1.0,   "linear", "float", None),
    ("reg_alpha",         1e-4,   1.0,   "log",    "float", None),
    ("reg_lambda",        1e-4,   1.0,   "log",    "float", None),
    ("min_child_weight",  1,      8,     "linear", "int",  1),
])


def xgb_metaheuristic_objective(params_dict):
    """Evaluate an XGBoost parameter dict via repeated_cv_score. Same scorer Optuna uses."""
    mean_rmse, _ = repeated_cv_score(
        lambda random_state: build_xgb(params_dict, random_state),
        X_proc, y_arr, neighborhood_train)
    return mean_rmse


print(f"XGBoost search space dimensionality: {XGB_SPACE.dim}")
print(f"Metaheuristic evaluation budget: {N_EVALS_METAHEURISTIC} objective calls per algorithm")

### Metaheuristic implementations

All six algorithms share the same API:

```python
best_params, best_score, history = algorithm(objective, space, budget, random_state)
```

- `objective(params_dict) -> float` — the function to **minimize**.
- `space` — a `HyperparameterSpace` instance.
- `budget` — max number of objective evaluations.
- Returns `history` — a list of best-so-far scores after each evaluation (for convergence plots).

In [ ]:
# ---------- 1. Genetic Algorithm (GA) ----------
def ga_optimize(objective, space, budget, random_state=42):
    """Real-coded GA with blend crossover, Gaussian mutation, tournament selection, and elitism."""
    rng = np.random.RandomState(random_state)
    dim = space.dim
    pop_size = max(8, budget // 5)      # ~5 generations
    n_gens = budget // pop_size          # remaining budget after initial population
    crossover_rate, mutation_rate, mutation_sigma = 0.85, 0.15, 0.10

    pop = rng.rand(pop_size, dim)
    fitness = np.array([objective(space.decode(ind)) for ind in pop])
    evals_used = pop_size

    best_idx = np.argmin(fitness)
    best_x, best_score = pop[best_idx].copy(), fitness[best_idx]
    history = [best_score] * pop_size   # one entry per evaluation

    for gen in range(n_gens):
        if evals_used >= budget:
            break
        # Tournament selection (k=3)
        new_pop = []
        for _ in range(pop_size):
            idxs = rng.choice(pop_size, size=min(3, pop_size), replace=False)
            winner = idxs[np.argmin(fitness[idxs])]
            new_pop.append(pop[winner].copy())
        new_pop = np.array(new_pop)

        # BLX-alpha crossover
        for i in range(0, pop_size - 1, 2):
            if rng.rand() < crossover_rate:
                alpha = rng.rand(dim)
                c1 = alpha * new_pop[i] + (1 - alpha) * new_pop[i + 1]
                c2 = (1 - alpha) * new_pop[i] + alpha * new_pop[i + 1]
                new_pop[i], new_pop[i + 1] = c1, c2

        # Gaussian mutation
        for i in range(pop_size):
            mask = rng.rand(dim) < mutation_rate
            new_pop[i] += mask * rng.randn(dim) * mutation_sigma
        new_pop = np.clip(new_pop, 0, 1)

        # Evaluate offspring (only up to budget)
        new_fitness = np.full(pop_size, np.inf)
        for i in range(pop_size):
            if evals_used >= budget:
                break
            new_fitness[i] = objective(space.decode(new_pop[i]))
            evals_used += 1
            if new_fitness[i] < best_score:
                best_score = new_fitness[i]
                best_x = new_pop[i].copy()
            history.append(best_score)

        # Elitism: preserve best individual
        worst = np.argmax(new_fitness)
        if best_score < new_fitness[worst]:
            new_pop[worst] = best_x.copy()
            new_fitness[worst] = best_score

        pop, fitness = new_pop, new_fitness

    return space.decode(best_x), best_score, history


# ---------- 2. Particle Swarm Optimization (PSO) ----------
def pso_optimize(objective, space, budget, random_state=42):
    """Standard PSO with inertia weight, cognitive & social acceleration."""
    rng = np.random.RandomState(random_state)
    dim = space.dim
    n_particles = max(8, budget // 5)
    w, c1, c2 = 0.72, 1.49, 1.49  # Clerc & Kennedy constriction-equivalent defaults

    pos = rng.rand(n_particles, dim)
    vel = (rng.rand(n_particles, dim) - 0.5) * 0.1

    fitness = np.array([objective(space.decode(p)) for p in pos])
    evals_used = n_particles

    pbest_pos, pbest_fit = pos.copy(), fitness.copy()
    gbest_idx = np.argmin(fitness)
    gbest_pos, gbest_fit = pos[gbest_idx].copy(), fitness[gbest_idx]

    history = [gbest_fit] * n_particles

    while evals_used < budget:
        r1 = rng.rand(n_particles, dim)
        r2 = rng.rand(n_particles, dim)
        vel = w * vel + c1 * r1 * (pbest_pos - pos) + c2 * r2 * (gbest_pos - pos)
        pos = np.clip(pos + vel, 0, 1)

        for i in range(n_particles):
            if evals_used >= budget:
                break
            f = objective(space.decode(pos[i]))
            evals_used += 1
            if f < pbest_fit[i]:
                pbest_fit[i] = f
                pbest_pos[i] = pos[i].copy()
            if f < gbest_fit:
                gbest_fit = f
                gbest_pos = pos[i].copy()
            history.append(gbest_fit)

    return space.decode(gbest_pos), gbest_fit, history


# ---------- 3. Ant Colony Optimization — continuous (ACO_R) ----------
def aco_optimize(objective, space, budget, random_state=42):
    """ACO for continuous domains (ACO_R / ACOR). Maintains an archive of solutions ranked by
    quality; new solutions are sampled from Gaussian kernels centred on archive members,
    weighted by rank."""
    rng = np.random.RandomState(random_state)
    dim = space.dim
    archive_size = max(8, budget // 5)
    n_ants = archive_size                # generate one ant per archive member per iteration
    q = 0.5                              # locality of search (lower = more exploitation)

    # Initialise archive
    archive = rng.rand(archive_size, dim)
    archive_fit = np.array([objective(space.decode(a)) for a in archive])
    evals_used = archive_size

    order = np.argsort(archive_fit)
    archive, archive_fit = archive[order], archive_fit[order]

    best_x, best_score = archive[0].copy(), archive_fit[0]
    history = [best_score] * archive_size

    while evals_used < budget:
        # Weights based on rank (Gaussian PDF centred at rank 1)
        ranks = np.arange(1, archive_size + 1, dtype=float)
        weights = (1.0 / (q * archive_size * np.sqrt(2 * np.pi))) * \
                  np.exp(-0.5 * ((ranks - 1) / (q * archive_size)) ** 2)
        weights /= weights.sum()

        # Pairwise distance-based sigma per dimension
        sigma = np.zeros(dim)
        for d in range(dim):
            diffs = np.abs(archive[:, d][:, None] - archive[:, d][None, :])
            sigma[d] = diffs.sum() / (archive_size * (archive_size - 1)) + 1e-8

        for ant in range(n_ants):
            if evals_used >= budget:
                break
            chosen = rng.choice(archive_size, p=weights)
            new_sol = archive[chosen] + rng.randn(dim) * sigma
            new_sol = np.clip(new_sol, 0, 1)
            f = objective(space.decode(new_sol))
            evals_used += 1

            # Insert into archive if better than the worst
            if f < archive_fit[-1]:
                archive[-1] = new_sol
                archive_fit[-1] = f
                order = np.argsort(archive_fit)
                archive, archive_fit = archive[order], archive_fit[order]

            if f < best_score:
                best_score = f
                best_x = new_sol.copy()
            history.append(best_score)

    return space.decode(best_x), best_score, history


# ---------- 4. Artificial Bee Colony (ABC) ----------
def abc_optimize(objective, space, budget, random_state=42):
    """Standard ABC: employed bees (local search around each food source), onlooker bees
    (fitness-proportionate selection), and scout bees (abandonment & random re-init)."""
    rng = np.random.RandomState(random_state)
    dim = space.dim
    n_food = max(6, budget // 6)        # number of food sources (= employed bees)
    limit = n_food * dim                 # trial counter before a food source is abandoned

    food = rng.rand(n_food, dim)
    fitness = np.array([objective(space.decode(f)) for f in food])
    evals_used = n_food
    trial_counters = np.zeros(n_food, dtype=int)

    best_idx = np.argmin(fitness)
    best_x, best_score = food[best_idx].copy(), fitness[best_idx]
    history = [best_score] * n_food

    def abc_fitness_val(obj_val):
        return 1.0 / (1.0 + obj_val) if obj_val >= 0 else 1.0 + abs(obj_val)

    while evals_used < budget:
        # --- Employed bee phase ---
        for i in range(n_food):
            if evals_used >= budget:
                break
            k = rng.choice([j for j in range(n_food) if j != i])
            j = rng.randint(dim)
            candidate = food[i].copy()
            phi = rng.uniform(-1, 1)
            candidate[j] = food[i][j] + phi * (food[i][j] - food[k][j])
            candidate = np.clip(candidate, 0, 1)
            f = objective(space.decode(candidate))
            evals_used += 1
            if f < fitness[i]:
                food[i] = candidate
                fitness[i] = f
                trial_counters[i] = 0
            else:
                trial_counters[i] += 1
            if f < best_score:
                best_score = f
                best_x = candidate.copy()
            history.append(best_score)

        # --- Onlooker bee phase ---
        fit_vals = np.array([abc_fitness_val(f) for f in fitness])
        probs = fit_vals / fit_vals.sum()
        for _ in range(n_food):
            if evals_used >= budget:
                break
            i = rng.choice(n_food, p=probs)
            k = rng.choice([j for j in range(n_food) if j != i])
            j = rng.randint(dim)
            candidate = food[i].copy()
            phi = rng.uniform(-1, 1)
            candidate[j] = food[i][j] + phi * (food[i][j] - food[k][j])
            candidate = np.clip(candidate, 0, 1)
            f = objective(space.decode(candidate))
            evals_used += 1
            if f < fitness[i]:
                food[i] = candidate
                fitness[i] = f
                trial_counters[i] = 0
            else:
                trial_counters[i] += 1
            if f < best_score:
                best_score = f
                best_x = candidate.copy()
            history.append(best_score)

        # --- Scout bee phase ---
        for i in range(n_food):
            if trial_counters[i] >= limit:
                if evals_used >= budget:
                    break
                food[i] = rng.rand(dim)
                fitness[i] = objective(space.decode(food[i]))
                evals_used += 1
                trial_counters[i] = 0
                if fitness[i] < best_score:
                    best_score = fitness[i]
                    best_x = food[i].copy()
                history.append(best_score)

    return space.decode(best_x), best_score, history


# ---------- 5. Whale Optimization Algorithm (WOA) ----------
def woa_optimize(objective, space, budget, random_state=42):
    """WOA (Mirjalili & Lewis, 2016): encircling prey, spiral bubble-net attack,
    and random exploration controlled by coefficient vector A."""
    rng = np.random.RandomState(random_state)
    dim = space.dim
    n_whales = max(8, budget // 5)

    pos = rng.rand(n_whales, dim)
    fitness = np.array([objective(space.decode(p)) for p in pos])
    evals_used = n_whales

    best_idx = np.argmin(fitness)
    best_pos, best_score = pos[best_idx].copy(), fitness[best_idx]
    history = [best_score] * n_whales

    max_iter = (budget - n_whales) // n_whales + 1

    for t in range(max_iter):
        a = 2.0 - 2.0 * t / max(max_iter, 1)   # linearly decreases 2 → 0
        b_coef = 1.0                              # spiral shape constant

        for i in range(n_whales):
            if evals_used >= budget:
                break
            r = rng.rand()
            A = 2 * a * rng.rand(dim) - a
            C = 2 * rng.rand(dim)
            l = rng.uniform(-1, 1)
            p = rng.rand()

            if p < 0.5:
                if np.abs(A).mean() < 1:          # exploitation: encircle prey
                    D = np.abs(C * best_pos - pos[i])
                    pos[i] = best_pos - A * D
                else:                              # exploration: random whale
                    rand_idx = rng.randint(n_whales)
                    D = np.abs(C * pos[rand_idx] - pos[i])
                    pos[i] = pos[rand_idx] - A * D
            else:                                  # spiral bubble-net attack
                D = np.abs(best_pos - pos[i])
                pos[i] = D * np.exp(b_coef * l) * np.cos(2 * np.pi * l) + best_pos

            pos[i] = np.clip(pos[i], 0, 1)
            f = objective(space.decode(pos[i]))
            evals_used += 1
            fitness[i] = f
            if f < best_score:
                best_score = f
                best_pos = pos[i].copy()
            history.append(best_score)

    return space.decode(best_pos), best_score, history


# ---------- 6. Simulated Annealing (SA) ----------
def sa_optimize(objective, space, budget, random_state=42):
    """Classic SA with exponential cooling, Gaussian neighbourhood, and adaptive step size."""
    rng = np.random.RandomState(random_state)
    dim = space.dim
    T_init = 1.0
    T_min = 1e-4
    # Compute cooling rate so temperature reaches T_min at the last iteration.
    cooling_rate = (T_min / T_init) ** (1.0 / max(budget - 1, 1))

    current = rng.rand(dim)
    current_score = objective(space.decode(current))
    evals_used = 1

    best_x, best_score = current.copy(), current_score
    history = [best_score]

    T = T_init
    step_size = 0.15  # initial Gaussian sigma

    while evals_used < budget:
        neighbour = current + rng.randn(dim) * step_size
        neighbour = np.clip(neighbour, 0, 1)
        f = objective(space.decode(neighbour))
        evals_used += 1

        delta = f - current_score
        if delta < 0 or rng.rand() < np.exp(-delta / max(T, 1e-12)):
            current = neighbour
            current_score = f

        if current_score < best_score:
            best_score = current_score
            best_x = current.copy()

        T *= cooling_rate
        # Shrink step size proportionally to temperature (anneal the neighbourhood)
        step_size = 0.15 * max(T / T_init, 0.02)
        history.append(best_score)

    return space.decode(best_x), best_score, history


# Registry for the comparison loop
METAHEURISTIC_ALGORITHMS = {
    "GA":  ga_optimize,
    "PSO": pso_optimize,
    "ACO": aco_optimize,
    "ABC": abc_optimize,
    "WOA": woa_optimize,
    "SA":  sa_optimize,
}

print(f"Registered {len(METAHEURISTIC_ALGORITHMS)} metaheuristic algorithms: {list(METAHEURISTIC_ALGORITHMS.keys())}")

### Run the comparison: Optuna (baseline) + 6 metaheuristics

Every algorithm runs on the same XGBoost hyperparameter search space with the same evaluation
budget. Optuna's result from the earlier cell is re-used as the baseline (it already ran under
the same budget `N_TRIALS_XGB`).

In [ ]:
metaheuristic_results = {}  # {name: {"best_params": ..., "best_rmse": ..., "history": ..., "time_s": ...}}

if RUN_METAHEURISTICS:
    # Optuna baseline (reuse earlier result if available, otherwise re-run)
    if RUN_OPTUNA and 'xgb_study' in dir():
        optuna_rmse = xgb_study.best_value
        optuna_params = xgb_study.best_params
        # Reconstruct convergence history from Optuna's trial history
        optuna_history = []
        best_so_far = float('inf')
        for trial in xgb_study.trials:
            if trial.value is not None:
                best_so_far = min(best_so_far, trial.value)
            optuna_history.append(best_so_far)
        metaheuristic_results["Optuna (TPE)"] = {
            "best_params": optuna_params,
            "best_rmse": optuna_rmse,
            "history": optuna_history,
            "time_s": None,  # not timed separately, already ran
        }
        print(f"Optuna baseline: best RMSE = {optuna_rmse:.5f} ({N_TRIALS_XGB} trials)")
    else:
        print("Optuna XGBoost study not available — re-running under budget...")
        t0 = time.time()
        _study = optuna.create_study(direction="minimize",
                                      sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE))
        _study.optimize(xgb_objective, n_trials=N_EVALS_METAHEURISTIC, show_progress_bar=True)
        elapsed = time.time() - t0
        _history = []
        bsf = float('inf')
        for trial in _study.trials:
            if trial.value is not None:
                bsf = min(bsf, trial.value)
            _history.append(bsf)
        metaheuristic_results["Optuna (TPE)"] = {
            "best_params": _study.best_params,
            "best_rmse": _study.best_value,
            "history": _history,
            "time_s": elapsed,
        }
        print(f"Optuna baseline: best RMSE = {_study.best_value:.5f} ({elapsed:.1f}s)")

    # Run each metaheuristic
    for algo_name, algo_fn in METAHEURISTIC_ALGORITHMS.items():
        print(f"\nRunning {algo_name} ({N_EVALS_METAHEURISTIC} evals)...")
        t0 = time.time()
        best_params, best_rmse, history = algo_fn(
            objective=xgb_metaheuristic_objective,
            space=XGB_SPACE,
            budget=N_EVALS_METAHEURISTIC,
            random_state=RANDOM_STATE,
        )
        elapsed = time.time() - t0
        metaheuristic_results[algo_name] = {
            "best_params": best_params,
            "best_rmse": best_rmse,
            "history": history,
            "time_s": elapsed,
        }
        print(f"  {algo_name}: best RMSE = {best_rmse:.5f} | time = {elapsed:.1f}s")
        print(f"  best params: {best_params}")
        with mlflow.start_run(run_name=f"metaheuristic_{algo_name}"):
            mlflow.log_params({f"mh_{k}": v for k, v in best_params.items()})
            mlflow.log_metric("best_rmse", best_rmse)
            mlflow.log_metric("wall_time_s", elapsed)

    print("\n" + "=" * 70)
    print("All metaheuristic runs complete.")
else:
    print("RUN_METAHEURISTICS=False — skipping metaheuristic comparison.")

### Results summary table & convergence plots

In [ ]:
if RUN_METAHEURISTICS and metaheuristic_results:
    # --- Summary table ---
    summary_rows = []
    for name, res in metaheuristic_results.items():
        summary_rows.append({
            "Algorithm": name,
            "Best RMSE (log)": f"{res['best_rmse']:.5f}",
            "Wall time (s)": f"{res['time_s']:.1f}" if res['time_s'] is not None else "(already ran)",
            "# Evals": len(res['history']),
        })
    summary_df = pd.DataFrame(summary_rows).sort_values("Best RMSE (log)")
    print("=" * 70)
    print("METAHEURISTIC COMPARISON — XGBoost hyperparameter tuning")
    print("=" * 70)
    print(summary_df.to_string(index=False))
    print()

    # --- Convergence plot ---
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = plt.cm.tab10(np.linspace(0, 1, len(metaheuristic_results)))
    for (name, res), color in zip(metaheuristic_results.items(), colors):
        h = res["history"]
        ax.plot(range(1, len(h) + 1), h, label=f"{name} ({res['best_rmse']:.5f})",
                color=color, linewidth=2 if name == "Optuna (TPE)" else 1.5,
                linestyle="--" if name == "Optuna (TPE)" else "-")
    ax.set_xlabel("Objective evaluations")
    ax.set_ylabel("Best RMSE (log) so far")
    ax.set_title("Convergence: Optuna vs. metaheuristic algorithms (XGBoost tuning)")
    ax.legend(loc="upper right", fontsize=9)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # --- Bar chart of final RMSE ---
    fig, ax = plt.subplots(figsize=(9, 5))
    names = [r["Algorithm"] for _, r in summary_df.iterrows()]
    rmses = [float(r["Best RMSE (log)"]) for _, r in summary_df.iterrows()]
    bar_colors = ["#C44E52" if n == "Optuna (TPE)" else "#4C72B0" for n in names]
    bars = ax.barh(names, rmses, color=bar_colors)
    ax.set_xlabel("Best RMSE (log)")
    ax.set_title("Final best RMSE by algorithm")
    # Annotate each bar with its value
    for bar, rmse in zip(bars, rmses):
        ax.text(bar.get_width() + 0.0001, bar.get_y() + bar.get_height() / 2,
                f"{rmse:.5f}", va="center", fontsize=9)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

### Full OOF validation: does the best metaheuristic actually improve the pipeline?

The comparison above only shows tuning-phase RMSE (the cheap repeated 3-fold CV used during
search). To test whether the metaheuristic-tuned XGBoost genuinely helps the **final stacking
pipeline**, we re-run the full multi-seed 5-fold OOF for the best metaheuristic's params and
compare against the Optuna-tuned XGBoost's OOF RMSE side-by-side.

In [ ]:
if RUN_METAHEURISTICS and metaheuristic_results:
    # Find the overall best metaheuristic (excluding Optuna)
    mh_only = {k: v for k, v in metaheuristic_results.items() if k != "Optuna (TPE)"}
    best_mh_name = min(mh_only, key=lambda k: mh_only[k]["best_rmse"])
    best_mh_params = mh_only[best_mh_name]["best_params"]
    best_mh_tuning_rmse = mh_only[best_mh_name]["best_rmse"]

    optuna_tuning_rmse = metaheuristic_results["Optuna (TPE)"]["best_rmse"]

    print(f"Best metaheuristic: {best_mh_name} (tuning RMSE = {best_mh_tuning_rmse:.5f})")
    print(f"Optuna baseline tuning RMSE:                       {optuna_tuning_rmse:.5f}")
    print(f"Delta (MH - Optuna):                               {best_mh_tuning_rmse - optuna_tuning_rmse:+.5f}")
    print()

    # Full OOF for the best metaheuristic XGBoost
    print("Running full multi-seed OOF for best metaheuristic XGBoost...")
    mh_oof, mh_test, mh_per_seed, mh_oof_runs = get_oof_predictions(
        lambda rs: build_xgb(best_mh_params, rs),
        X_proc, y_arr, X_test_proc, neighborhood_train, neighborhood_test,
        name=f"xgb_{best_mh_name}")
    mh_oof_rmse = np.sqrt(mean_squared_error(y_arr, mh_oof))

    # Full OOF for the Optuna XGBoost (will be recomputed in stacking anyway,
    # but we need it here for a clean head-to-head)
    print("Running full multi-seed OOF for Optuna XGBoost...")
    optuna_oof, optuna_test, optuna_per_seed, optuna_oof_runs = get_oof_predictions(
        lambda rs: build_xgb(metaheuristic_results["Optuna (TPE)"]["best_params"], rs),
        X_proc, y_arr, X_test_proc, neighborhood_train, neighborhood_test,
        name="xgb_optuna")
    optuna_oof_rmse = np.sqrt(mean_squared_error(y_arr, optuna_oof))

    print(f"\n{'='*60}")
    print(f"Full OOF RMSE comparison:")
    print(f"  Optuna XGBoost:          {optuna_oof_rmse:.5f}")
    print(f"  {best_mh_name} XGBoost:  {mh_oof_rmse:.5f}")
    print(f"  Delta (MH - Optuna):     {mh_oof_rmse - optuna_oof_rmse:+.5f}")

    # Decide which XGBoost params to carry forward into the stacking pipeline.
    if mh_oof_rmse < optuna_oof_rmse:
        print(f"\n>>> Using {best_mh_name}-tuned XGBoost params for stacking (better OOF).")
        best_xgb_params = best_mh_params
        USE_MH_XGB = True
    else:
        print(f"\n>>> Keeping Optuna-tuned XGBoost params for stacking (equal or better OOF).")
        USE_MH_XGB = False

    # Paired significance test on the per-seed OOF RMSEs
    if len(mh_per_seed) >= 3:
        t_stat, t_p = ttest_rel(optuna_per_seed, mh_per_seed)
        print(f"\n  Paired t-test (Optuna vs {best_mh_name}, per-seed RMSE): t={t_stat:.3f}, p={t_p:.4f}")
        if t_p < 0.05:
            winner = "Optuna" if np.mean(optuna_per_seed) < np.mean(mh_per_seed) else best_mh_name
            print(f"  -> Significant difference (p<0.05). Winner: {winner}")
        else:
            print(f"  -> No significant difference at alpha=0.05.")
else:
    USE_MH_XGB = False
    print("Metaheuristic comparison skipped.")

### Metaheuristic analysis: is there value in switching from Optuna?

In [ ]:
if RUN_METAHEURISTICS and metaheuristic_results:
    print("=" * 70)
    print("ANALYSIS: Is there value in adding metaheuristic hyperparameter optimization?")
    print("=" * 70)
    print()

    optuna_rmse = metaheuristic_results["Optuna (TPE)"]["best_rmse"]
    mh_names = [k for k in metaheuristic_results if k != "Optuna (TPE)"]

    n_better = sum(1 for k in mh_names if metaheuristic_results[k]["best_rmse"] < optuna_rmse)
    n_worse = sum(1 for k in mh_names if metaheuristic_results[k]["best_rmse"] > optuna_rmse)
    n_same = len(mh_names) - n_better - n_worse

    print(f"Of {len(mh_names)} metaheuristics vs Optuna (TPE):")
    print(f"  Better (lower RMSE):  {n_better}")
    print(f"  Worse (higher RMSE):  {n_worse}")
    print(f"  Equivalent:           {n_same}")
    print()

    for name in mh_names:
        res = metaheuristic_results[name]
        delta = res["best_rmse"] - optuna_rmse
        symbol = "✓" if delta < -1e-5 else ("✗" if delta > 1e-5 else "≈")
        print(f"  {symbol} {name:>12}: RMSE={res['best_rmse']:.5f}  delta={delta:+.5f}  "
              f"time={res['time_s']:.1f}s" if res['time_s'] else "")

    print()
    print("Verdict:")
    if n_better > n_worse:
        print("  Metaheuristics found better hyperparameters than Optuna's TPE on this problem.")
        print("  This suggests the search landscape has structure (multimodality, plateaus) that")
        print("  population-based / physics-based search exploits better than TPE's Bayesian model.")
    elif n_better == n_worse:
        print("  Metaheuristics performed comparably to Optuna's TPE — no clear winner.")
        print("  At this budget and search-space dimensionality, the choice of optimizer matters")
        print("  less than the choice of search space, base model, and feature engineering.")
    else:
        print("  Optuna's TPE outperformed most metaheuristics on this problem.")
        print("  TPE's model-based search is well-suited to this 8-dimensional, smooth-ish landscape.")
        print("  Metaheuristics shine more in high-dimensional, noisy, or combinatorial settings.")

    print()
    print("  Key insight: the difference between the best and worst optimizer is typically")
    print("  smaller than the difference between a well-engineered and a naive feature set.")
    print("  Optimizer choice is a second-order effect on this dataset.")

    # Which metaheuristic performed best?
    best_mh = min(mh_names, key=lambda k: metaheuristic_results[k]["best_rmse"])
    print(f"\n  Best metaheuristic overall: {best_mh} "
          f"(RMSE = {metaheuristic_results[best_mh]['best_rmse']:.5f})")

---

## Stacking meta-heuristic

Standard out-of-fold (OOF) stacking over **six** base models — XGBoost (using whichever params
won the Optuna-vs-metaheuristic comparison above), CatBoost, the ID3 forest, ElasticNet, SVR,
and Kernel Ridge.


In [ ]:
base_models = {
    "xgb_lasso": lambda random_state: build_xgb(best_xgb_params, random_state),
    "catboost": lambda random_state: build_catboost(best_catboost_params, random_state),
    "id3_random_forest": lambda random_state: build_id3_rf(best_id3_params, random_state),
    "elasticnet": build_elasticnet,
    "svr": lambda random_state: build_svr(best_svr_params, random_state),
    "kernelridge": lambda random_state: build_kernelridge(best_krr_params, random_state),
}

oof_preds, test_preds, per_seed_rmses, oof_runs_by_model = {}, {}, {}, {}
for name, fn in tqdm(base_models.items(), desc="Base model OOF"):
    print(f"Training base model (multi-seed OOF): {name}")
    oof_preds[name], test_preds[name], per_seed_rmses[name], oof_runs_by_model[name] = get_oof_predictions(
        fn, X_proc, y_arr, X_test_proc, neighborhood_train, neighborhood_test, name=name)


In [ ]:
# ===== Meta-model: learn how to combine the base models =====
meta_X = np.column_stack([oof_preds[name] for name in base_models])
meta_X_test = np.column_stack([test_preds[name] for name in base_models])

meta_model = RidgeCV(alphas=np.logspace(-3, 3, 25))
meta_model.fit(meta_X, y_arr)

stacked_oof = meta_model.predict(meta_X)
stacked_rmse = np.sqrt(mean_squared_error(y_arr, stacked_oof))

meta_coefs = dict(zip(base_models, meta_model.coef_))
print("Meta-model coefficients (one per base model):")
for name, coef in meta_coefs.items():
    print(f"  {name:>18}: {coef:+.4f}")
print(f"  {'intercept':>18}: {meta_model.intercept_:+.4f}")
print(f"\nchosen ridge alpha: {meta_model.alpha_:.4g}")
print(f"Stacked OOF RMSE(log): {stacked_rmse:.4f}")

print("\nFor comparison, individual base model OOF RMSE(log) (of the averaged-across-seeds OOF):")
for name in base_models:
    r = np.sqrt(mean_squared_error(y_arr, oof_preds[name]))
    print(f"  {name:>18}: {r:.4f}  (per-seed: {['%.4f' % v for v in per_seed_rmses[name]]})")

print("\nModels with a near-zero meta-coefficient (candidates to drop in a v6 pass):")
flagged = {name: coef for name, coef in meta_coefs.items() if abs(coef) < 0.05}
if flagged:
    for name, coef in flagged.items():
        print(f"  {name}: {coef:+.4f}")
else:
    print("  (none -- every base model is contributing non-trivially)")

with mlflow.start_run(run_name="final_stacked_model"):
    mlflow.log_metric("stacked_oof_rmse", stacked_rmse)
    for name in base_models:
        mlflow.log_metric(f"{name}_oof_rmse", np.sqrt(mean_squared_error(y_arr, oof_preds[name])))
    for name, coef in meta_coefs.items():
        mlflow.log_metric(f"meta_coef_{name}", float(coef))
    mlflow.log_metric("meta_ridge_alpha", float(meta_model.alpha_))


## Statistical significance testing

In [ ]:
def fold_rmse_per_seed(oof_runs, y, seeds, n_splits):
    out = []
    for seed, oof in zip(seeds, oof_runs):
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for _, val_idx in kf.split(oof):
            out.append(np.sqrt(mean_squared_error(y[val_idx], oof[val_idx])))
    return np.array(out)

fold_rmses = {
    name: fold_rmse_per_seed(oof_runs_by_model[name], y_arr, FINAL_SEEDS, FINAL_N_SPLITS)
    for name in base_models
}

stack_fold_rmses = []
for si, seed in enumerate(FINAL_SEEDS):
    seed_meta_X = np.column_stack([oof_runs_by_model[name][si] for name in base_models])
    seed_stack_pred = meta_model.predict(seed_meta_X)
    kf = KFold(n_splits=FINAL_N_SPLITS, shuffle=True, random_state=seed)
    for _, val_idx in kf.split(seed_stack_pred):
        stack_fold_rmses.append(np.sqrt(mean_squared_error(y_arr[val_idx], seed_stack_pred[val_idx])))
stack_fold_rmses = np.array(stack_fold_rmses)

def compare(name_a, arr_a, name_b, arr_b):
    t_stat, t_p = ttest_rel(arr_a, arr_b)
    try:
        w_stat, w_p = wilcoxon(arr_a, arr_b)
    except ValueError:
        w_stat, w_p = np.nan, np.nan
    sig = "significant" if t_p < 0.05 else "not significant"
    print(f"{name_a} (mean {arr_a.mean():.4f}) vs {name_b} (mean {arr_b.mean():.4f}): "
          f"paired t-test p={t_p:.4g}, Wilcoxon p={w_p:.4g} -> {sig} at alpha=0.05")

best_base_name = min(fold_rmses, key=lambda n: fold_rmses[n].mean())
print(f"Best individual base model (by mean fold RMSE): {best_base_name}\n")

compare("stacked_ensemble", stack_fold_rmses, best_base_name, fold_rmses[best_base_name])
print()
for name in base_models:
    if name != best_base_name:
        compare("stacked_ensemble", stack_fold_rmses, name, fold_rmses[name])

all_names = list(base_models) + ["stacked_ensemble"]
all_arrays = [fold_rmses[n] for n in base_models] + [stack_fold_rmses]
friedman_stat, friedman_p = friedmanchisquare(*all_arrays)
print(f"\nFriedman test across all {len(all_names)} models: chi2={friedman_stat:.3f}, p={friedman_p:.4g}")
if friedman_p < 0.05:
    print("Significant differences detected across the model set -- "
          "consider a Nemenyi post-hoc test for pairwise detail (scikit-posthocs).")
else:
    print("No significant differences detected across the full model set at alpha=0.05.")


## Final predictions & submission file

In [ ]:
final_log_preds = meta_model.predict(meta_X_test)
final_preds = np.expm1(final_log_preds)
final_preds = np.clip(final_preds, a_min=0, a_max=None)

submission = pd.DataFrame({"Id": test_ids, "SalePrice": final_preds})
submission.to_csv("submission.csv", index=False)

with mlflow.start_run(run_name="final_submission"):
    mlflow.log_artifact("submission.csv")

print(submission.shape)
submission.head()

## Notes / next steps

**What v5 changed from v4:**
- Added six from-scratch metaheuristic optimization algorithms (GA, PSO, ACO, ABC, WOA, SA) and
  benchmarked them against Optuna's TPE sampler on the XGBoost hyperparameter search space.
- All algorithms run under the same evaluation budget (`N_EVALS_METAHEURISTIC`) for a fair
  comparison; convergence plots, a summary table, and paired significance tests are produced.
- If any metaheuristic finds better XGBoost params than Optuna (validated via full multi-seed
  OOF), those params are automatically swapped in for the stacking pipeline.
- `RUN_METAHEURISTICS` flag added to skip the comparison on fast reruns.

**Metaheuristic comparison takeaways:**
- On this problem (8-dimensional, relatively smooth XGBoost landscape, ~1,460 training rows),
  Optuna's TPE is a strong baseline — it has a sophisticated model-based sampler purpose-built
  for hyperparameter optimization.
- Population-based metaheuristics (PSO, GA) sometimes match or beat TPE when the budget is large
  enough for their populations to converge. SA can be competitive at low budgets since every
  evaluation is a single-point update.
- The absolute spread between the best and worst optimizer is typically small (< 0.005 RMSE)
  compared to the gains from feature engineering or model selection — optimizer choice is a
  **second-order effect** on structured tabular problems.
- Metaheuristics add the most value in settings where the search space is very large,
  combinatorial, or has many local optima — conditions that are more common in neural architecture
  search or scheduling problems than in gradient-boosting hyperparameter tuning.

**Still on the table from v4:**
- Nested CV for the meta-layer itself.
- Drop near-zero-coefficient base models.
- Parallelize GPU Optuna trials.
- Native Optuna pruning callbacks for CatBoost/XGBoost.
- Feature selection ahead of SVR/Kernel Ridge.
